# The Time Diversification Effect

In [ ]:
import pandas as pd
import numpy as np
import random
from matplotlib import pyplot as plt
import seaborn as sns
from datetime import date
from dateutil.relativedelta import relativedelta

from trading_algos import optimization as tao
from trading_algos import datasets as tad
from trading_algos import plots as tap
from trading_algos import utils as tau
from trading_algos.utils import head_tail as ht

%load_ext autoreload
%autoreload 2

## Load Data

In [ ]:
# Load a complete collection of S&P500 stocks from repository
# Not interested in survivors here
df_stocks = tad.get_sp500(survivors=False)

In [ ]:
ht(df_stocks)

In [ ]:
df_stocks.shape

In [ ]:
print(df_stocks.columns.get_level_values(1).nunique(), 'stocks')

## Simulating Portfolios

In [ ]:
log_prices = np.log(df_stocks.Close)
ht(log_prices)

In [ ]:
# 3 Month Rolling Returns
rolling_log_return = log_prices - log_prices.shift(63)
ht(rolling_log_return)

In [ ]:
df_agg = rolling_log_return.agg({'mean', 'std'}).rename({'mean': 'Return', 'std': 'Risk'}).T
ht(df_agg)

In [ ]:
lq = df_agg.quantile(0.25)
uq = df_agg.quantile(0.75)
iqr = uq - lq
lo = lq - 1.5*iqr
uo = uq + 1.5*iqr

In [ ]:
df_outliers = df_agg[((df_agg > uo) | (df_agg < lo)).any(axis=1)].sort_values(['Risk', 'Return'], ascending=False)
outliers = df_outliers.index.to_list()
df_outliers.shape

In [ ]:
df_outliers.head()

In [ ]:
fig, ax = plt.subplots(1,2)

sns.boxplot(df_agg[['Risk']], ax=ax[0])
sns.boxplot(df_agg[['Return']], ax=ax[1])

In [ ]:
ncols = 5
nrows = int(np.ceil(len(outliers)/ncols))


fig, axes = plt.subplots(nrows, ncols, figsize=(12, 30), tight_layout=True)

fig.tight_layout(pad=2, rect=[0,0.05,1,0.95])
fig.suptitle(f'Log Price Trend of Outlier Stocks', fontsize=16)

for i, ticker in enumerate(outliers):

    ax = axes[int(i/5), i%5]
    ax.set_title(f"{ticker}")
    ax.set_yticklabels([])
    ax.set_yticks([])
    ax.set_ylabel(' ')
    ax.set_xticklabels([])
    ax.set_xticks([])
    ax.set_xlabel(' ')
    ax.spines[['top','bottom','left','right']].set_visible(False)

    ax.plot((log_prices[ticker]), linewidth=0.5)

# Determine how many empty plots there are on the last row of the figure
num_empty_plots = nrows * ncols - len(outliers)
if num_empty_plots > 0:
    for i in range(1, num_empty_plots + 1):
        # Hide empty plots
        axes[nrows - 1, ncols - i].set_axis_off()

In [ ]:
# For the sake of this notebook I will drop all outliers
# Usually would investigate further to decide which ones need dropping
log_prices = log_prices.drop(columns=outliers)

In [ ]:
df_agg = df_agg.drop(outliers)

In [ ]:
fig, ax = plt.subplots(1,2)

sns.boxplot(df_agg[['Risk']], ax=ax[0])
sns.boxplot(df_agg[['Return']], ax=ax[1])

In [ ]:
years = []
cagr = []
risk = []
prob_beat_rf = []
cagr_min = []
cagr_max = []
cagr_median = []
cagr_5pct = []
cagr_95pct = []

for i in range(12,360+1, 12):
    
    print(i, 'Months')

    ann_rolling_log_return = (log_prices - log_prices.shift(i * 21)) / (i / 12)
    
    # print('Years: ', i, ' CAGR: ', cagr_rolling.mean().mean(), ' Risk: ', cagr_rolling.std().mean())
    years.append(i/12)
    cagr.append(ann_rolling_log_return.mean().mean())
    risk.append(ann_rolling_log_return.std().mean())
    prob_beat_rf.append(((ann_rolling_log_return > 0.04 )* 1).mask(ann_rolling_log_return.isna(), np.nan).mean().mean())
    cagr_min.append(ann_rolling_log_return.min().mean())
    cagr_max.append(ann_rolling_log_return.max().mean())
    cagr_median.append(ann_rolling_log_return.median().mean())
    cagr_5pct.append(ann_rolling_log_return.quantile(0.05).mean())
    cagr_95pct.append(ann_rolling_log_return.quantile(0.95).mean())


df_inv = pd.DataFrame({'Year':years, 
                       'Return':cagr, 
                       'Risk':risk, 
                       'Prob Beat RF':prob_beat_rf,
                       'Min Return': cagr_min,
                       'Max Return': cagr_max,
                       'Median Return': cagr_median,
                       '5th Percentile': cagr_5pct,
                       '95th Percentile': cagr_95pct
                       }).set_index('Year')
df_inv

In [ ]:
ann_rolling_log_return

In [ ]:
df_inv.iloc[-1]

In [ ]:
ann_rolling_log_return.mean().mean()

In [ ]:
ann_rolling_log_return.describe(percentiles=[0.05, 0.95]).mean(axis=1).to_frame().T

In [ ]:
df_inv['Prob Beat RF'].plot()

In [ ]:
df_inv[['5th Percentile', 'Median Return', '95th Percentile']].plot()

In [ ]:
(df_inv['Return'] / df_inv['Risk']).plot()

In [ ]:
df_inv = df_inv.reset_index()

In [ ]:
(np.exp(df_inv['Return'] * (df_inv['Year']/12)) - 1) * 100

In [ ]:
(df_inv['Return']-0.04)

In [ ]:
((ann_rolling_log_return > 0 )* 1).mask(ann_rolling_log_return.isna(), np.nan).mean().mean()